In [ ]:
%pip install requests

In [ ]:
import requests

URL = 'https://finance.naver.com/sise/sise_quant.nhn'

response = requests.get(URL, params={
    # 쿼리파라미터
    'sosok': 0
}, headers={
    # 요청 헤더
}, data={
    # 요청 본문 1 - form data / www/x-form-urlencoded
}, json={
    # 요청 본문 2 - json 포맷 / application/json
})
# res, r

# requests.post()
# requests.put()
# requests.delete()

In [12]:
# 상태 코드
response.status_code

200

In [ ]:
# 응답 헤더
dict(response.headers)

In [ ]:
# 응답 본문
print(response.text.find('선물인버스'))
# 데이터가 HTML문서 안에 포함되어있는 경우 - 정적 웹

36418


In [ ]:
# 파이썬 기본 요청 헤더

# User-Agent: python-requests/2.11.0 -> Bot으로 오해받기 딱 좋음 (차단 기본 설정)

headers = {
    'User-Agent': 'Mozilla 5.0' # 어? 사람(브라우저)인가?
}

requests.get('', params={}, headers=headers)

In [ ]:
# 선택자 활용해서 HTML문서 파싱 도와주는 라이브러리
%pip install beautifulsoup4

In [19]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, 'html.parser')

In [ ]:
# 태그 선택자: 태그명
soup.select('title')

[<title>거래상위 종목 : Npay 증권</title>]

In [ ]:
# ID 선택자: #
soup.select_one('#contentarea')

In [ ]:
# class 선택자: .
soup.select('.no') # [0]

In [28]:
# Find
soup.find("title") # 가장위의것하나

<title>거래상위 종목 : Npay 증권</title>

In [30]:
soup.find("li", class_="type1") # class 로좁히기

<li class="type1 lst1_1"><a class="off" href="/sise/sise_index.naver?code=KOSPI" onclick="clickcr(this,'siu.1','','',event);"><span class="blind">코스피</span></a></li>

In [ ]:
soup.find(id="contentarea") # id 로찾기

In [ ]:
result = soup.find("a", attrs={"class":"thumb"}) # 못 찾으면 None
print(result)

None


In [34]:
soup.find_all("a", attrs={"class":"thumb"})

[]

In [ ]:
soup.find_all("li") # 조건에맞는것전부(리스트

In [ ]:
soup.select("a.tltle") # 리스트

In [ ]:
soup.select_one("#contentarea") # 하나

In [37]:
result = soup.select("a.tltle")

print(result[0].text) # select 는여러개를가져옵니다

KODEX 200선물인버스2X


In [ ]:
soup.select('a[href^="/item/main.naver?code"]')

In [44]:
result[0]

<a class="tltle" href="/item/main.naver?code=252670">KODEX 200선물인버스2X</a>

In [ ]:
# 텍스트
result[0].text

'KODEX 200선물인버스2X'

In [ ]:
# 속성
result[0].attrs['class']
result[0].attrs['href']

'/item/main.naver?code=252670'

In [45]:
# 좁혀 들어가기— 찾은 객체에서 다시 찾습니다
box = soup.find(id="contentarea")
# 재검색 가능
table = box.find("table", class_="type_2")

In [ ]:
# CSS 선택자 
#contentarea table.type_2

In [ ]:
soup.select_one('#contentarea table.type_2')

In [51]:
trs = soup.select('#contentarea table.type_2 tr')

len(trs[0].select('td')) < 7

True

In [ ]:
trs[2].select('td')

In [ ]:
# 데이터가 들어가있는 한 줄들 (반복문 박스 대상)
trs = soup.select('#contentarea table.type_2 tr')

stocks = []
for tr in trs:
    # 데이터가 없는 줄은 날리기
    tds = tr.select('td')
    if len(tds) < 7:
        continue

    # 전일비 처리
    direction = tds[3].select_one('span.blind').text.strip()
    amount = tds[3].select_one('span.tah').text.strip()

    stocks.append({
        '종목명': tds[1].text.strip(),
        '현재가': tds[2].text.strip(),
        # '전일비': tds[3].text.strip().replace('\t', '').replace('\n', ' '),
        '전일비': direction + ' ' + amount,
        '등락률': tds[4].text.strip(),
        '거래량': tds[5].text.strip(),
    })

In [ ]:
stocks

In [ ]:
URL = 'https://finance.naver.com/sise/sise_quant.nhn'

def fetch() -> str:
    response = requests.get(URL, params={'sosok': 0})
    return response.text

html = fetch()

def parse(html: str) -> list[dict]:
    soup = BeautifulSoup(html, 'html.parser')
    trs = soup.select('#contentarea table.type_2 tr')
    stocks = []
    for tr in trs:
        tds = tr.select('td')
        if len(tds) < 7:
            continue
        direction = tds[3].select_one('span.blind').text.strip()
        amount = tds[3].select_one('span.tah').text.strip()

        stocks.append({
            '종목명': tds[1].text.strip(),
            '현재가': tds[2].text.strip(),
            '전일비': direction + ' ' + amount,
            '등락률': tds[4].text.strip(),
            '거래량': tds[5].text.strip(),
        })
    return stocks

stocks = parse(html)

### 예제

- 최신공고 크롤링-공고제목, 카테고리, 상태와 onclick의goView(‘12345’)를 크롤링하여 정리

In [ ]:
URL = 'https://youth.seoul.go.kr/infoData/sprtInfo/list.do'

headers = {
    # user-agent
}

res = requests.get(URL, headers=headers)

res.status_code

200

In [ ]:
# 데이터가 있다 -> 정적 웹 의심
# 없다? -> 인코딩(한글로 검색했다면,) / 동적 웹 의심
res.text.find('서울청년센터 중구')

116111

In [62]:
soup = BeautifulSoup(res.text, 'html.parser')

In [64]:
items = soup.select('div.feed-item')

In [ ]:
result = []

def get_text(tag) -> str:
    return tag.text.strip() if tag else ''

for item in items:
    # 데이터 정리
    # 공고 제목, 카테고리, 상태, onclick
    result.append({
        '제목': get_text( item.select_one('.name') ),
        '카테고리': get_text( item.select_one('.cate') ),
        '상태': get_text( item.select_one('.state') ),
        'onclick': item.select_one('a').attrs['onclick'],
    })

result

[{'제목': '서울광역청년센터 X 서울청년센터 <2026 CJ제일제당 나눔냉장고 캠페인🍚>',
  '카테고리': '생활지원',
  '상태': '모집중',
  'onclick': "goView('73003');"},
 {'제목': '📢 프로그램 게시요청 가이드 📢',
  '카테고리': '',
  '상태': '상시',
  'onclick': "goView('68721');"},
 {'제목': '서울청년센터 중구 <8월 티톡 : 나의 취향을 굴리는 시간> 모집',
  '카테고리': '커뮤니티',
  '상태': '모집중',
  'onclick': "goView('73040');"},
 {'제목': '강남구 1인가구 커뮤니티센터 <T라 논리적인, F라 감성적인> 참여자 모집',
  '카테고리': '마음건강',
  '상태': '모집중',
  'onclick': "goView('72922');"},
 {'제목': '강남구 1인가구 커뮤니티센터 <면접관이 알려주는 합격비법!> 참여자 모집',
  '카테고리': '진로',
  '상태': '모집중',
  'onclick': "goView('72920');"},
 {'제목': '강남구 1인가구 커뮤니티센터 <AI기반 뮤직비디오 제작> 참여자 모집',
  '카테고리': '문화/예술',
  '상태': '모집중',
  'onclick': "goView('72888');"},
 {'제목': '강남구 1인가구 커뮤니티센터 <나도 생명지킴이?! 강남구 자살예방 안전망> 참여자 모집',
  '카테고리': '마음건강',
  '상태': '모집중',
  'onclick': "goView('72865');"},
 {'제목': '강남구 1인가구 커뮤니티센터 <퇴근길 인문학> 참여자 모집',
  '카테고리': '마음건강',
  '상태': '모집중',
  'onclick': "goView('72859');"}]